# Property Values and Property Value Management Tool (PVMT) support

capellambse provides access to property values and property value groups, as well as the additional information maintained by the Property Value Management Tool (PVMT) extension.

Let's first take a look at how to work with basic property values.

In [1]:
import capellambse

model = capellambse.MelodyModel(
    "../../../tests/data/melodymodel/5_0/Melody Model Test.aird"
)

Model objects can own property values and PV groups. To access those, use the `property_values` and `property_value_groups` attributes respectively:

In [2]:
obj = model.search("LogicalComponent").by_name("Whomping Willow")

In [3]:
obj.property_values

[0] <IntegerPropertyValue 'cars_defeated' (a928fa22-cef7-4357-9b87-675a432f6591)>

In [4]:
obj.property_values[0]

applied_property_value_groups,(Empty list)
applied_property_values,(Empty list)
constraints,(Empty list)
description,
diagrams,(Empty list)
enumerations,(Empty list)
filtering_criteria,(Empty list)
layer,"LogicalArchitecture ""Logical Architecture"" (853cb005-cba0-489b-8fe3-bb694ad4543b)"
name,cars_defeated
parent,"""Whomping Willow"" (3bdd4fa2-5646-44a1-9fa6-80c68433ddb7)"
progress_status,NOT_SET


In [5]:
obj.property_value_groups

[0] <PropertyValueGroup 'Stats' (81bcc3d3-24d2-411e-a296-9943029acfd9)>

## dict-like access to property values

In addition to standard attribute-based access, these property value-related attributes can also behave like dicts in some cases. Here the dict key equates to the `name` of a property value or group, and the dict value equates to either the `value` of a property value object, or a list of PV objects in the group (which again can behave dict-like).

This significantly shortens the way to a specific value. For comparison, this would be the "usual" attribute-based way of accessing it:

In [6]:
obj.property_value_groups.by_name("Stats").property_values.by_name("WIS").value

150

And here is the same again, leveraging the dict-like behavior:

In [7]:
obj.property_value_groups["Stats"]["WIS"]

150

This of course also works for ungrouped property values:

In [8]:
obj.property_values["cars_defeated"]

1

In [9]:
obj.property_value_groups["Stats"]

[0] <IntegerPropertyValue 'HP' (0e95950d-272f-4b50-b68d-ee8fed002c94)>
[1] <IntegerPropertyValue 'STR' (32be6f26-4b33-4861-b501-3bfce3df94cb)>
[2] <IntegerPropertyValue 'AGI' (addb1e4a-9ecc-411c-b7ef-b76388e5840d)>
[3] <IntegerPropertyValue 'WIS' (7c8c9c21-86b9-4dba-9c7c-cd9db2f09c11)>
[4] <IntegerPropertyValue 'INT' (647a5565-a1de-4e15-ab21-eb628eea413c)>

In [10]:
obj.property_value_groups["Stats"]["WIS"]

150

These property values can also be written to:

In [11]:
obj.property_value_groups["Stats"]["INT"] = 18
obj.property_value_groups["Stats"]

[0] <IntegerPropertyValue 'HP' (0e95950d-272f-4b50-b68d-ee8fed002c94)>
[1] <IntegerPropertyValue 'STR' (32be6f26-4b33-4861-b501-3bfce3df94cb)>
[2] <IntegerPropertyValue 'AGI' (addb1e4a-9ecc-411c-b7ef-b76388e5840d)>
[3] <IntegerPropertyValue 'WIS' (7c8c9c21-86b9-4dba-9c7c-cd9db2f09c11)>
[4] <IntegerPropertyValue 'INT' (647a5565-a1de-4e15-ab21-eb628eea413c)>

# Property Value Management (PVMT)

Property values become much more powerful when using the [Property Value Management Tool extension](https://github.com/eclipse-capella/capella/wiki/PVMT). This extension groups property values into domains and groups, and allows to automatically apply certain groups based on specific conditions. These conditions may include the object class, the architectural layer, and other already applied property values.

Metadata about the defined domains and groups is stored in a package named "EXTENSIONS" at the root level of the model, which is conveniently accessible as `model.pvmt`. The package will automatically be created if it does not exist yet.

In [12]:
model.pvmt

From there, we can easily enumerate the defined domains:

In [13]:
model.pvmt.domains

AssertionError: 

[0] <ManagedDomain 'DarkMagic'>

And see which groups are defined in that domain:

In [14]:
domain = model.pvmt.domains["DarkMagic"]
domain.groups

[0] <ManagedGroup 'Power'>
[1] <ManagedGroup 'Power Level'>

In [15]:
group = domain.groups["Power Level"]
group

Group definitions also have a `selector`, which contains rules that determine the applicable model elements.

The current ruleset can be introspected by reading the attributes of the `selector`:

In [16]:
print(group.selector.classes)
print(group.selector.layers)
print(group.selector.properties)

()
(<class 'capellambse.metamodel.la.LogicalArchitecture'>,)
(('DarkMagic.Power.Max', '<', '10000.0'),)


The ManagedGroup object provides some helper methods that interpret this selector:

In [17]:
help(group.find_applicable)

Help on method find_applicable in module capellambse.extensions.pvmt._config:

find_applicable() -> 'm.ElementList' method of capellambse.extensions.pvmt._config.ManagedGroup instance
    Find all elements in the model that this group applies to.



In [18]:
group.find_applicable()

[0] <LogicalComponent 'Prof. A. P. W. B. Dumbledore' (08e02248-504d-4ed8-a295-c7682a614f66)>
[1] <LogicalComponent 'Prof. S. Snape' (6f463eed-c77b-4568-8078-beec0536f243)>
[2] <LogicalComponent 'Harry J. Potter' (a8c46457-a702-41c4-a971-c815c4c5a674)>
[3] <LogicalComponent 'R. Weasley' (ff7b8672-84db-4b93-9fea-22a410907fb1)>

In [19]:
help(group.applies_to)

Help on method applies_to in module capellambse.extensions.pvmt._config:

applies_to(obj: 'm.ModelObject') -> 'bool' method of capellambse.extensions.pvmt._config.ManagedGroup instance
    Determine whether this group applies to a model element.



In [20]:
dumbledore = model.search("LogicalComponent").by_name(
    "Prof. A. P. W. B. Dumbledore"
)
group.applies_to(dumbledore)

True

And finally, a group can be applied to an element with the `apply` method. If the group was already applied earlier, the existing group instance will be returned:

In [21]:
group.apply(dumbledore)

applied_property_value_groups,"PropertyValueGroup ""Power Level"" (a0cb5a23-955e-43d6-a633-2c4e66991364)"
applied_property_values,(Empty list)
constraints,(Empty list)
description,
diagrams,(Empty list)
filtering_criteria,(Empty list)
layer,"LogicalArchitecture ""Logical Architecture"" (853cb005-cba0-489b-8fe3-bb694ad4543b)"
name,DarkMagic.Power Level
parent,"""Prof. A. P. W. B. Dumbledore"" (08e02248-504d-4ed8-a295-c7682a614f66)"
progress_status,NOT_SET
property_value_groups,(Empty list)


Each model element can have several property value groups applied to it. PVMT-managed groups can be easily inspected by using the element's `pvmt` attribute:

In [22]:
dumbledore.pvmt

TypeError: Map function must return a model element or a list of model elements, not <capellambse.extensions.pvmt._config.ManagedGroup object at 0x783b4b91a870>

TypeError: Map function must return a model element or a list of model elements, not <capellambse.extensions.pvmt._config.ManagedGroup object at 0x783b4b921fd0>

The `groupdefs` attribute will give access to the definitions of all groups that apply to this element:

In [23]:
dumbledore.pvmt.groupdefs

TypeError: Map function must return a model element or a list of model elements, not <capellambse.extensions.pvmt._config.ManagedGroup object at 0x783b4b922870>

On the other hand, the `applied_groups` attribute will yield the group instances that are currently applied to the object.

**Note:** This list may be missing some groups from `groupdefs`. These can be applied using the group definition's `apply` method (see above).

In [24]:
dumbledore.pvmt.applied_groups

[0] <PropertyValueGroup 'DarkMagic.Power' (42dc9aeb-a06d-4012-8bc6-02fb06f94cdc)>
[1] <PropertyValueGroup 'DarkMagic.Power Level' (6cf12387-7187-4b91-9efe-8fedb8a148cb)>

Property values can also be accessed using convenient dict-like semantics.

With keys like "Domain.Group", an applied group instance will be returned, where the group is applied first if needed:

In [25]:
dumbledore.pvmt["DarkMagic.Power"]

applied_property_value_groups,"PropertyValueGroup ""Power"" (1e241678-282c-46f3-abbd-e0cf34e743ee)"
applied_property_values,(Empty list)
constraints,(Empty list)
description,
diagrams,(Empty list)
filtering_criteria,(Empty list)
layer,"LogicalArchitecture ""Logical Architecture"" (853cb005-cba0-489b-8fe3-bb694ad4543b)"
name,DarkMagic.Power
parent,"""Prof. A. P. W. B. Dumbledore"" (08e02248-504d-4ed8-a295-c7682a614f66)"
progress_status,NOT_SET
property_value_groups,(Empty list)


Keys may also contain a Property, in which case the value of that property is returned directly.
This is similar to the basic property values shown in the first part of this notebook.
However, like in the case without a Property, the group will first be applied if needed.
Newly applied groups have all their properties set to the default values from the group definition.

In [26]:
dumbledore.pvmt["DarkMagic.Power.Max"]

1600.0

This latter style also supports directly setting individual properties.

In [27]:
dumbledore.pvmt["DarkMagic.Power.Max"] = 9000
dumbledore.pvmt["DarkMagic.Power.Max"]

9000.0